<a href="https://colab.research.google.com/github/sardorbeks4/cis3120/blob/main/CIS3120_Sardorbek_Shorobov_LibraryDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#step 1 & 2
import sqlite3
import csv
import urllib.request
import os



# GitHub raw URLs for the three CSV files

# Replace <username> and <repo> with the actual values provided by your instructor

BASE_URL = "https://raw.githubusercontent.com/sardorbeks4/cis3120/main/"


BOOK_URL   = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL   = BASE_URL + "Loan.csv"


# Local paths in the Colab /content directory

BOOK_PATH   = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH   = "/content/Loan.csv"

DB_PATH = "/content/library.db"
print("Setup Complete")

for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

print(os.listdir("/content"))

Setup Complete
Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv
['.config', 'Member.csv', 'Book.csv', 'Loan.csv', 'sample_data']


In [8]:
#step 3 Create tables
if os.path.exists(DB_PATH):
  os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')
# CREATE TABLE statements go here
conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall())

Tables created.
[('Book',), ('Loan',), ('Member',)]


In [9]:
#step 4 load books data
import csv

with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

Book rows loaded: 11


In [10]:
#step 5 load member data
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


In [11]:
#step 6 load loan data
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None

        conn.execute(
            """
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?);
            """,
            (
                row["callNo"],
                int(row["id"]),
                row["dateBorrowed"],
                date_returned,
                row["dateDue"]
            )
        )

conn.commit()
print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Loan rows loaded: 4


**Retrieve all columns from the Book Table, ordered alphabetically**

In [13]:
#Query 1:

query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

rows = conn.execute(query1).fetchall()

print("Query 1 Results:")
for row in rows:
    print(row)

Query 1 Results:
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


**Retrieve the title of each book and the first and last name of the member who borrowed it for all loans where the book has not yet been returned.**

In [14]:
#Query 2:
query2 = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

rows = conn.execute(query2).fetchall()

print("Query 2 Results:")
for row in rows:
    print(row)

Query 2 Results:
("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


**Retrieve the full loan history for the book with callNo 'R 141 E45 2006', showing the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.**

In [15]:
#Query 3:

query3 = """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC;
"""

rows = conn.execute(query3).fetchall()

print("Query 3 Results:")
for row in rows:
    print(row)

Query 3 Results:
('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


**Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table.**

In [16]:
#Query 4:

query4 = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

rows = conn.execute(query4).fetchall()

print("Query 4 Results:")
for row in rows:
    print(row)

Query 4 Results:
(4, 'John', 'Martin')


**Retrieve each member's full name and the total number of loans they have made, including members with zero loans. Order by number of loans descending.**

In [17]:
#Query 5:

query5 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.id) AS total_loans
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY total_loans DESC;
"""

rows = conn.execute(query5).fetchall()

print("Query 5 Results:")
for row in rows:
    print(row)

Query 5 Results:
('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


**Business question: Which authors currently have books checked out?**


In [18]:
#Query 6:

query6 = """
SELECT Book.author, COUNT(*) AS books_currently_checked_out
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
WHERE Loan.dateReturned IS NULL
GROUP BY Book.author
ORDER BY books_currently_checked_out DESC, Book.author;
"""

rows = conn.execute(query6).fetchall()

print("Query 6 Results:")
for row in rows:
    print(row)

Query 6 Results:
('Joe Celko', 1)
('Lynne Elliott', 1)


**Markdown Summary**

One data quality issue in this dataset is that the `dateReturned` field is blank for books that have not yet been returned, so the empty values had to be converted to `NULL` when loading the Loan table. Another issue is that the date fields are stored as text rather than as true date values, which limits more advanced date analysis. A limitation of this dataset is that it is very small and does not represent many features of a real library system, such as multiple copies of the same title, overdue fines, book categories, reservations, or branch locations. Even so, the dataset is useful for demonstrating relational design, foreign keys, joins, and aggregation in SQLite.